Import Libraries

In [18]:
# Core data handling
import pandas as pd
import numpy as np

# Country to region mapping
import pycountry_convert as pc

# File handling
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)

Loading the Dataset

In [19]:
# Loading the data

df = pd.read_csv(r"C:\Users\PC\Desktop\Future Interns\FUTURE_DS_01\online_retail.csv")


Initial Data Inspection

Understanding the structure and quality of the dataset


In [20]:
# Showing the first five rows 
print("First 5 rows of the dataset: \t")
display(df.head())

# Displaying the total number of rows and columns
print("\n The Online retail dataset has", df.shape[0], "rows and", df.shape[1], "columns")

# Checking the names of the columns
print("\n Columns in the dataset:")
print(df.columns.tolist())


First 5 rows of the dataset: 	


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom



 The Online retail dataset has 541909 rows and 8 columns

 Columns in the dataset:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [21]:
# Dataset structure
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [22]:
# Missing values
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [23]:
# Duplicate records
df.duplicated().sum()

5268

Data Cleaning

Handling Missing Values

Removing rows where critical fields are missing.

In [24]:
df = df.dropna(subset=["CustomerID", "Description"])

Fix Data Types

Convert columns to appropriate formats.

In [25]:
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    format="mixed",
    errors="coerce"
)

df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

Remove Invalid Data

Drop rows with failed conversions or unrealistic values.

In [26]:
df = df.dropna(subset=["InvoiceDate", "Quantity", "UnitPrice"])

# Remove returns / invalid transactions
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

Future Engineering

In [27]:
# Creating Revenue Column
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

Extract Date Features

In [28]:
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["Day"] = df["InvoiceDate"].dt.day

Region Mapping

Converting country names into broader geographic regions for regional analysis

In [35]:
# Counts of each country
df["Country"].value_counts()

Country
United Kingdom          354321
Germany                   9040
France                    8341
EIRE                      7236
Spain                     2484
Netherlands               2359
Belgium                   2031
Switzerland               1841
Portugal                  1462
Australia                 1182
Norway                    1071
Italy                      758
Channel Islands            748
Finland                    685
Cyprus                     614
Sweden                     451
Austria                    398
Denmark                    380
Poland                     330
Japan                      321
Israel                     248
Unspecified                244
Singapore                  222
Iceland                    182
USA                        179
Canada                     151
Greece                     145
Malta                      112
United Arab Emirates        68
European Community          60
RSA                         57
Lebanon                     45


In [34]:
# All countries in the country column
df["Country"].unique()

array(['United Kingdom', 'France', 'Australia', 'Netherlands', 'Germany',
       'Norway', 'EIRE', 'Switzerland', 'Spain', 'Poland', 'Portugal',
       'Italy', 'Belgium', 'Lithuania', 'Japan', 'Iceland',
       'Channel Islands', 'Denmark', 'Cyprus', 'Sweden', 'Finland',
       'Austria', 'Greece', 'Singapore', 'Lebanon',
       'United Arab Emirates', 'Israel', 'Saudi Arabia', 'Czech Republic',
       'Canada', 'Unspecified', 'Brazil', 'USA', 'European Community',
       'Bahrain', 'Malta', 'RSA'], dtype=object)

In [36]:
# Custom mapping for non-standard country names
CUSTOM_REGION_MAP = {
    "Channel Islands": "EU",
    "EIRE": "EU",
    "Unspecified": "EU", # Since EU is the dorminant
    "European Community": "EU",
    "RSA": "AF"   # Correct continent code for South Africa
}


def country_to_region(country_name: str) -> str:
    """Convert country name to continent/region with custom overrides."""
    
    # Handle missing or invalid values
    if pd.isna(country_name):
        return "Unknown"
    
    country_name = str(country_name).strip()

    # Step 1: Check custom mappings first
    if country_name in CUSTOM_REGION_MAP:
        return CUSTOM_REGION_MAP[country_name]

    # Step 2: Fallback to pycountry_convert
    try:
        country_code = pc.country_name_to_country_alpha2(country_name)
        continent = pc.country_alpha2_to_continent_code(country_code)
        return continent
    except Exception:
        return "Unknown"


# Apply mapping
df["Region"] = df["Country"].apply(country_to_region)

Final Dataset Check

In [37]:
print("Final Shape:", df.shape)
df.head()

Final Shape: (397884, 13)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,Year,Month,Day,Region
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,12,1,EU
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,EU
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,12,1,EU
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,EU
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,EU


Export the cleaned dataset for visualization on the Power BI Desktop 

In [38]:
# Exporting the final cleaned data to excel
online_retail_df = df.copy()

online_retail_df.to_csv(
    r"C:\Users\PC\Desktop\Future Interns\FUTURE_DS_01\online_retail_data.csv",
    index=False
)